In [2]:
%%bash
echo "📦 1. 检测到系统重启，正在重新为你构建数字分身锻造炉..."
pip install --upgrade pip
pip install unsloth_zoo
pip install --no-deps "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
pip install --no-deps xformers trl peft accelerate bitsandbytes
echo "✅ 环境全部重装完毕！请继续运行下一步！"

📦 1. 检测到系统重启，正在重新为你构建数字分身锻造炉...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 112.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 21.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 87.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: fsspec
    Found existing installat

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-_pdxsrpj/unsloth_c0cec61a6b4548718b6a5bc85a8eb7d5


In [1]:
# 【极其重要】：在 Kaggle 原生 Cell 里，unsloth 必须是第一行！
from unsloth import FastVisionModel, is_bfloat16_supported
import torch
import os
import shutil
import trl
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainerCallback

# =====================================================================
# 1. 战报监听器（原生 Cell 里，print 会瞬间上屏，绝不延迟）
# =====================================================================
class StepLoggerCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            step = state.global_step
            max_steps = state.max_steps
            loss = logs.get("loss", 0)
            lr = logs.get("learning_rate", 0)
            print(f"🎯 [微调雷达] Step: {step}/{max_steps} | 损失(Loss): {loss:.4f} | 学习率: {lr:.6f}")

# =====================================================================
# 2. 核心路径配置
# =====================================================================
DATASET_PATH = "/kaggle/input/datasets/xiangjiaowei/dierbuwancheng/dataset_jh_gold (3).jsonl"
OUTPUT_MODEL_DIR = "/kaggle/working/Qwen3-VL-4B-JH-Twin-HF"
ZIP_OUTPUT_PATH = "/kaggle/working/Qwen3-VL-4B-JH-Twin-HF"

print("📥 1. 正在加载基座模型 Qwen3-VL-4B-Instruct (大概需要1-2分钟下载，请稍等)...")
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "Qwen/Qwen3-VL-4B-Instruct",
    load_in_4bit = True,
    use_gradient_checkpointing = "unsloth", 
)

print("🧠 2. 配置 PEFT 权重，屏蔽视觉层...")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = False,      
    finetune_language_layers = True,     
    finetune_attention_modules = True,   
    r = 16, lora_alpha = 16, lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

print(f"📊 3. 正在读取数据集: {DATASET_PATH} ...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

def format_chat_template(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

print("🔄 正在对齐语料格式 (这步会很快)...")
dataset = dataset.map(format_chat_template, batched=True)
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
if text_tokenizer.pad_token is None: text_tokenizer.pad_token = text_tokenizer.eos_token

print("⚙️ 4. 正在注入训练参数...")
if hasattr(trl, "SFTConfig"):
    from trl import SFTConfig
    training_args = SFTConfig(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4, warmup_steps = 10,
        num_train_epochs = 1, learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.01, lr_scheduler_type = "linear",
        seed = 3407, output_dir = "outputs", report_to = "none",              
        max_seq_length = 2048, dataset_text_field = "text", packing = False, dataset_num_proc = 2,
    )
    trainer = SFTTrainer(model = model, train_dataset = dataset, processing_class = text_tokenizer, args = training_args, callbacks=[StepLoggerCallback()])
else:
    from transformers import TrainingArguments
    training_args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4, warmup_steps = 10,
        num_train_epochs = 1, learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(), bf16 = is_bfloat16_supported(),
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.01, lr_scheduler_type = "linear",
        seed = 3407, output_dir = "outputs", report_to = "none",
    )
    trainer = SFTTrainer(model = model, train_dataset = dataset, dataset_text_field = "text", max_seq_length = 2048, tokenizer = text_tokenizer, args = training_args, callbacks=[StepLoggerCallback()])

print("🚀 5. 微调正式启动！进度条和战报马上出现！")
trainer.train()

print("📦 6. 训练完成！正在合并为标准 16-bit 完整权重...")
model.save_pretrained_merged(OUTPUT_MODEL_DIR, tokenizer, save_method="merged_16bit")
print("✅ 权重合并保存成功！")

print("🧹 7. [防爆盘] 删除官方底层基座缓存，腾出空间...")
shutil.rmtree("/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-4B-Instruct", ignore_errors=True)

print("🗜️ 8. 正在打包成 ZIP 压缩文件 (大概需要几分钟，请不要关闭网页)...")
shutil.make_archive(ZIP_OUTPUT_PATH, 'zip', OUTPUT_MODEL_DIR)

print("🧹 9. 压缩完毕！清理原文件夹...")
shutil.rmtree(OUTPUT_MODEL_DIR, ignore_errors=True)

print("\n✨ 【大功告成】请前往右侧面板 /kaggle/working 下载 Qwen3-VL-4B-JH-Twin-HF.zip！")


ModuleNotFoundError: No module named 'unsloth'